# Profile D-MPNN Training

This notebook profiles a representative D-MPNN training step using `torch.profiler`.

- On CUDA, it records both CPU and CUDA activities.
- On Apple Silicon / MPS, PyTorch profiler support is more limited, so this notebook records CPU activity and still helps surface host-side bottlenecks and synchronization points.


In [1]:
import os
os.chdir("..")

import random
from pathlib import Path

import torch
from torch.profiler import profile, record_function, ProfilerActivity, schedule

from dmpnn.synthetic import generate_unique_graphs
from dmpnn.model import DMPNN
from dmpnn.training import DMPNNTrainer
from dmpnn.graph_utils import iter_batches, prepare_batch

In [21]:
SEED = 42
NUM_GRAPHS = 20000
BATCH_SIZE = 512
WARMUP_STEPS = 5
PROFILE_STEPS = 20

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")

train_graphs, _ = generate_unique_graphs(
    NUM_GRAPHS,
    min_nodes=10,
    max_nodes=30,
    seed=SEED,
)

model = DMPNN(
    node_feat_dim=4,
    edge_feat_dim=2,
    hidden_dim=512,
    num_steps=3,
    head_hidden_size=512,
    output_size=1,
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

trainer = DMPNNTrainer(
    model,
    optimizer=optimizer,
    loss_fn=torch.nn.MSELoss(),
    device=device,
    task_type="regression",
)

batches = iter_batches(train_graphs, batch_size=BATCH_SIZE, shuffle=False)
batches = iter(batches)

# Warmup
for _ in range(WARMUP_STEPS):
    graphs = next(batches)
    batch = prepare_batch(graphs, device)
    trainer.train_batch(batch)

print(f"Completed {WARMUP_STEPS} warmup steps.")

activities = [ProfilerActivity.CPU]
if device == "cuda":
    activities.append(ProfilerActivity.CUDA)

with profile(
    activities=activities,
    schedule=schedule(wait=1, warmup=1, active=PROFILE_STEPS, repeat=1),
    record_shapes=True,
    profile_memory=True,
    with_stack=True,
) as prof:
    for step in range(PROFILE_STEPS + 2):
        with record_function("prepare_batch"):
            graphs = next(batches)
            batch = prepare_batch(graphs, device)

        with record_function("dmpnn_train_batch"):
            trainer.train_batch(batch)

        prof.step()

print("Profiling complete.")

sort_key = "self_cuda_time_total" if device == "cuda" else "self_cpu_time_total"
print(prof.key_averages().table(sort_by=sort_key, row_limit=30))

Using device: cuda
Completed 5 warmup steps.
Profiling complete.
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                               aten::mm         1.13%      13.833ms         1.44%      17.635ms      44.088us      78.512ms        60.48%      78.512ms    